In [31]:
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split

In [32]:
# Configuration
RAVDESS_DIR = "RAVDESS"
EMOTIONS_DIR = "Emotions"
SCREAM_DIR = "Screaming"
ARDUINO_DIR = "Arduino"

OUTPUT_CSV = "aggregated_audio_dataset.csv"

SEED = 42

AGGRESSIVE = "aggressive"
NEUTRAL = "neutral"

In [33]:
def add_record(records, path, source, label, metadata=None):

    if metadata is None:
        metadata = {}

    row = {
        "filepath": path,
        "source": source,
        "label": label
    }

    row.update(metadata)

    records.append(row)

In [34]:
def process_ravdess(records):

    files = glob.glob(
        os.path.join(RAVDESS_DIR, "*.wav")
    )

    emotion_map = {
        "01": "neutral",
        "02": "calm",
        "03": "happy",
        "04": "sad",
        "05": "angry",
        "06": "fearful",
        "07": "disgust",
        "08": "surprised"
    }

    intensity_map = {
        "01": "normal",
        "02": "strong"
    }

    for filepath in files:

        filename = os.path.basename(filepath)

        parts = filename.replace(".wav", "").split("-")

        if len(parts) < 7:
            continue

        emotion_code = parts[2]
        intensity_code = parts[3]

        emotion = emotion_map.get(emotion_code)
        intensity = intensity_map.get(intensity_code)

        if emotion == "angry":
            label = AGGRESSIVE

        elif emotion == "fearful" and intensity == "strong":
            label = AGGRESSIVE

        else:
            label = NEUTRAL

        add_record(
            records,
            filepath,
            source="RAVDESS",
            label=label,
            metadata={
                "emotion": emotion,
                "intensity": intensity
            }
        )

def process_emotions_dataset(records):

    files = glob.glob(
        os.path.join(EMOTIONS_DIR, "*.wav")
    )

    for filepath in files:

        filename = os.path.basename(filepath)

        emotion = filename.split("-")[-1]
        emotion = emotion.replace(".wav", "").lower()

        label = NEUTRAL

        add_record(
            records,
            filepath,
            source="EMOTIONS_DATASET",
            label=label,
            metadata={
                "emotion": emotion
            }
        )

def process_screaming_dataset(records):

    scream_dir = os.path.join(SCREAM_DIR, "Screaming")
    non_scream_dir = os.path.join(SCREAM_DIR, "NotScreaming")

    scream_files = glob.glob(
        os.path.join(scream_dir, "**/*.wav"),
        recursive=True
    )

    for filepath in scream_files:

        add_record(
            records,
            filepath,
            source="SCREAMING_DATASET",
            label=AGGRESSIVE,
            metadata={
                "screaming": True
            }
        )

    non_scream_files = glob.glob(
        os.path.join(non_scream_dir, "**/*.wav"),
        recursive=True
    )

    for filepath in non_scream_files:

        add_record(
            records,
            filepath,
            source="SCREAMING_DATASET",
            label=NEUTRAL,
            metadata={
                "screaming": False
            }
        )

def process_arduino_audio(records):

    label_map = {
        "Aggressive": AGGRESSIVE,
        "Neutral": NEUTRAL
    }

    for folder_name, label in label_map.items():

        label_dir = os.path.join(ARDUINO_DIR, folder_name)

        files = glob.glob(
            os.path.join(label_dir, "**/*.wav"),
            recursive=True
        )

        for filepath in files:

            add_record(
                records,
                filepath,
                source="ARDUINO",
                label=label
            )

In [35]:
def print_dataset_diagnostics(df):

    print("\n================ DATASET DIAGNOSTICS ================\n")

    print("Total samples:")
    print(len(df))

    print("\nLabel distribution:")
    print(df["label"].value_counts())

    print("\nLabel percentages:")
    print((df["label"].value_counts(normalize=True) * 100).round(2))

    print("\nSource distribution:")
    print(df["source"].value_counts())

    print("\nSplit distribution:")
    print(df["split"].value_counts())

    print("\n=====================================================\n")

In [36]:
def build_dataset():

    records = []

    print("Processing RAVDESS...")
    process_ravdess(records)

    print("Processing emotions dataset...")
    process_emotions_dataset(records)

    print("Processing screaming dataset...")
    process_screaming_dataset(records)

    print("Processing Arduino audio...")
    process_arduino_audio(records)

    if len(records) == 0:
        raise ValueError("No audio files found.")

    # Create dataframe
    df = pd.DataFrame(records)

    # Shuffle
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

    train_df, temp_df = train_test_split(
        df,
        test_size=0.2,
        random_state=SEED,
        stratify=df["label"]
    )

    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.5,
        random_state=SEED,
        stratify=temp_df["label"]
    )

    train_df["split"] = "train"
    val_df["split"] = "validation"
    test_df["split"] = "test"

    final_df = pd.concat([
        train_df,
        val_df,
        test_df
    ]).reset_index(drop=True)

    # Save CSV
    final_df.to_csv(OUTPUT_CSV, index=False)

    print(f"\nSaved dataset manifest to: {OUTPUT_CSV}")

    print_dataset_diagnostics(final_df)

    return final_df

In [37]:
dataset_df = build_dataset()

Processing RAVDESS...
Processing emotions dataset...
Processing screaming dataset...
Processing Arduino audio...

Saved dataset manifest to: aggregated_audio_dataset.csv

================ DATASET DIAGNOSTICS ================

Total samples:
5001

Label distribution:
label
neutral       3851
aggressive    1150
Name: count, dtype: int64

Label percentages:
label
neutral       77.0
aggressive    23.0
Name: proportion, dtype: float64

Source distribution:
source
SCREAMING_DATASET    3493
RAVDESS              1440
EMOTIONS_DATASET       68
Name: count, dtype: int64

Split distribution:
split
train         4000
test           501
validation     500
Name: count, dtype: int64


